# 07 — Agent RAG (Multi-Agent)

```
╔══════════════════════════════════════════════════════════════════════════╗
║                    7. AGENT RAG (MULTI-AGENT)                            ║
║                                                                          ║
║                        TOOLS & INTEGRATIONS                              ║
║                                                                          ║
║  Query ──► AI Agent ──► AI Agent (Retriever-1) ──► Vector Search A ─►  Slack║
║            (Planner)                                                     ║
║               │    ──► AI Agent (Retriever-2) ──► Vector Search B ─►  Gmail║
║               │                                                          ║
║               │    ──► AI Agent (Retriever-N) ──► Web Search ─────►  Web APIs║
║               │                │                                         ║
║               └────────────────┘ aggregate                              ║
║                                   │                                      ║
║  Response ◄── Generative Model ◄──┘                                     ║
╚══════════════════════════════════════════════════════════════════════════╝
```

## What is Multi-Agent RAG?

NB06 showed a single router agent that picks one tool at a time. Multi-agent RAG scales this by:

1. **Planner agent**: decomposes the complex query into sub-questions
2. **Specialised retriever agents**: each with their own tool, knowledge base, and search strategy
3. **Parallel dispatch**: retriever agents can run concurrently
4. **Aggregator/synthesiser**: combines results into a coherent answer

This mirrors how a team of specialists would research a complex question: the manager (Planner) assigns tasks, specialists (Retrievers) work in parallel, then the manager synthesises.

## Dataset's external tool integrations

The infographic shows real integrations (Slack, Gmail, Web APIs). In this notebook:
- All external tools are **deterministic mocks** so the notebook runs fully offline
- Each mock is documented with exactly what a real implementation would do
- This lets you focus on the orchestration pattern, not the plumbing

## When is this worth the complexity?

**Use multi-agent when**:
- Queries span multiple independent knowledge domains
- Retrieval from different sources can happen in parallel
- You need external real-time data alongside your knowledge base

**Don't use when**:
- All data is in one place
- Queries are simple and uniform
- Latency is critical (orchestration overhead is significant)

In [ ]:
import sys; sys.path.insert(0, '..')
import ragkit.config as cfg
import json
import time
import concurrent.futures

cfg.BACKEND = "claude"   # "claude" | "local"
print(f"Backend: {cfg.BACKEND}  |  Device: {cfg.DEVICE}")

## Step 1 — Build specialised knowledge bases

In a real deployment these would be separate services. Here, each retriever agent gets a filtered collection.

In [ ]:
from ragkit.data import build_chunked_corpus
from ragkit.vectorstore import build_collection, query_collection
from rank_bm25 import BM25Okapi
import re

texts, metadatas = build_chunked_corpus(chunk_size=200, overlap=40)

def tokenise(text):
    return re.findall(r'[a-z0-9]+(?:[\-\.][a-z0-9]+)*', text.lower())

# ── Retriever A: Product Specs ─────────────────────────────────────────────
spec_texts = [t for t, m in zip(texts, metadatas) if m['category'] == 'spec']
spec_metas = [m for m in metadatas if m['category'] == 'spec']
spec_collection = build_collection("helios_multi_spec", spec_texts, spec_metas, persist_dir="../.chroma")
spec_bm25 = BM25Okapi([tokenise(t) for t in spec_texts])

# ── Retriever B: Incidents & Procedures ───────────────────────────────────
ops_texts = [t for t, m in zip(texts, metadatas) if m['category'] in ('incident', 'proc', 'faq')]
ops_metas = [m for m in metadatas if m['category'] in ('incident', 'proc', 'faq')]
ops_collection = build_collection("helios_multi_ops", ops_texts, ops_metas, persist_dir="../.chroma")
ops_bm25 = BM25Okapi([tokenise(t) for t in ops_texts])

# ── Retriever C: People & Projects ───────────────────────────────────────
org_texts = [t for t, m in zip(texts, metadatas) if m['category'] in ('team', 'project')]
org_metas = [m for m in metadatas if m['category'] in ('team', 'project')]
org_collection = build_collection("helios_multi_org", org_texts, org_metas, persist_dir="../.chroma")

print(f"Spec KB:      {spec_collection.count()} chunks  (product docs, firmware, datasheets)")
print(f"Ops KB:       {ops_collection.count()} chunks  (incidents, procedures, FAQ)")
print(f"Org KB:       {org_collection.count()} chunks  (people, teams, projects)")

## Step 2 — Mock external tool integrations

Each mock has a comment explaining what the real implementation would do.

In [ ]:
def mock_web_search(query: str) -> str:
    """
    REAL IMPLEMENTATION:
      import requests
      response = requests.get("https://api.search.brave.com/res/v1/web/search",
                              headers={"Accept": "application/json", "X-Subscription-Token": API_KEY},
                              params={"q": query, "count": 5})
      return response.json()
    Or: use Tavily, SerpAPI, Google Custom Search API.
    """
    # Deterministic mock based on query content
    if "helios" in query.lower() or "robot" in query.lower():
        return """Web search results (mock):
[1] Helios Robotics announces HeliosArm V3 development — 7-DOF, 25kg payload (TechCrunch, 2024-03)
[2] Collaborative robots market: ISO/TS 15066 compliance overview (IEEE Spectrum, 2024)
[3] Helios Robotics wins 'Best Industrial Robot 2023' award (RoboticsBusiness Review)"""
    return "Web search results (mock): No relevant web results found for this query."

def mock_slack_search(channel: str, query: str) -> str:
    """
    REAL IMPLEMENTATION:
      from slack_sdk import WebClient
      client = WebClient(token=SLACK_BOT_TOKEN)
      results = client.search_messages(query=query, channel=channel)
      return results["messages"]
    """
    if "joint 4" in query.lower() or "j4" in query.lower():
        return """Slack #fleet-alerts (mock):
[2024-10-15 09:32] @carlos.mendoza: Client AutoFab flagging J4 alarm again at 42°C ambient,
  they've applied the speed limit but want to know when FW-V2-2.3.2 ships.
[2024-10-15 10:14] @sasha.ivanova: FW-V2-2.3.2 target date Dec 1. Thermal pad kit HR-FIX-J4-THERM
  available now as interim hardware fix."""
    return "Slack search (mock): No recent messages matching this query."

def mock_servicenow_lookup(incident_id: str) -> str:
    """
    REAL IMPLEMENTATION:
      import pysnow
      client = pysnow.Client(instance=SN_INSTANCE, user=SN_USER, password=SN_PASS)
      incident = client.resource(api_path='/table/incident')
      return incident.get(query={'number': incident_id}).one()
    """
    records = {
        "INC-2024-031": "Status: Open | Severity: High | Assigned: Sasha Ivanova | ETA: FW-V2-2.3.2 Dec 1 2024",
        "INC-2024-019": "Status: Resolved | Fixed: FW-M1-1.5.1 (Jul 2024) | Affected: 23 units",
        "INC-2024-011": "Status: Resolved | Fixed: HR-REED-02 upgrade kit | All units before 2024-06 affected",
    }
    return records.get(incident_id, f"ServiceNow (mock): No record found for {incident_id}")

print("External tool mocks defined:")
print("  - mock_web_search(query)")
print("  - mock_slack_search(channel, query)")
print("  - mock_servicenow_lookup(incident_id)")

# Quick test
print("\nServiceNow lookup test:")
print(mock_servicenow_lookup("INC-2024-031"))

## Step 3 — Define the specialised retriever agents

Each retriever agent has:
- A **role** (what it specialises in)
- A set of **tools** it can use
- A **knowledge base** it searches

In [ ]:
from ragkit.llm import generate, generate_tools

def retriever_specs(sub_question: str) -> dict:
    """Retriever-1: Product specifications, part numbers, firmware."""
    # Try BM25 first for part numbers
    tokens = tokenise(sub_question)
    bm25_scores = spec_bm25.get_scores(tokens)
    top_bm25 = bm25_scores.max()
    
    if top_bm25 > 1.0:  # BM25 found a good exact match
        top_idx = bm25_scores.argsort()[::-1][:3]
        results = [spec_texts[i] for i in top_idx if bm25_scores[i] > 0]
        strategy = "bm25"
    else:  # Fall back to semantic
        hits = query_collection(spec_collection, sub_question, k=3)
        results = [h.text for h in hits]
        strategy = "semantic"
    
    return {
        "agent": "Retriever-Specs",
        "strategy": strategy,
        "context": "\n\n".join(results),
    }

def retriever_ops(sub_question: str) -> dict:
    """Retriever-2: Incidents, procedures, field service notes."""
    # Check for incident ID pattern
    import re as re_mod
    inc_pattern = re_mod.search(r'INC-\d{4}-\d{3}', sub_question)
    
    results = []
    strategy = "semantic"
    
    if inc_pattern:
        # Direct ServiceNow lookup
        inc_id = inc_pattern.group(0)
        sn_result = mock_servicenow_lookup(inc_id)
        results.append(f"ServiceNow: {sn_result}")
        strategy = "servicenow"
    
    # Also semantic search
    hits = query_collection(ops_collection, sub_question, k=3)
    results.extend(h.text for h in hits)
    
    return {
        "agent": "Retriever-Ops",
        "strategy": strategy + "+semantic",
        "context": "\n\n".join(results[:4]),
    }

def retriever_org(sub_question: str) -> dict:
    """Retriever-3: People, teams, projects, escalation paths."""
    hits = query_collection(org_collection, sub_question, k=3)
    
    # Also check Slack for recent team activity
    slack_result = mock_slack_search("#fleet-alerts", sub_question)
    
    context = "\n\n".join(h.text for h in hits)
    if "No recent" not in slack_result:
        context += f"\n\nSlack channel context:\n{slack_result}"
    
    return {
        "agent": "Retriever-Org",
        "strategy": "semantic+slack",
        "context": context,
    }

def retriever_web(sub_question: str) -> dict:
    """Retriever-N: External web search for recent news and external context."""
    result = mock_web_search(sub_question)
    return {
        "agent": "Retriever-Web",
        "strategy": "web_search",
        "context": result,
    }

ALL_RETRIEVERS = {
    "specs":   (retriever_specs,  "Product specs, part numbers (HR-XXX), firmware (FW-XXX), datasheets"),
    "ops":     (retriever_ops,    "Incidents (INC-YYYY-NNN), procedures, field service, FAQ"),
    "org":     (retriever_org,    "People, teams, projects, contacts, organisational info"),
    "web":     (retriever_web,    "External web search for news, standards, external context"),
}

print("Retriever agents:")
for name, (fn, desc) in ALL_RETRIEVERS.items():
    print(f"  {name:8s} — {desc}")

## Step 4 — The Planner agent

The Planner decomposes the user query into sub-questions and assigns each to the right retriever.

In [ ]:
PLANNER_SYSTEM = """You are a retrieval planner for Helios Robotics.

Available retriever agents:
- specs: Product specs, part numbers (HR-XXX), firmware versions (FW-XXX), datasheets
- ops: Incidents (INC-YYYY-NNN), procedures, field service, FAQ
- org: People, teams, projects, escalation contacts, org info
- web: External web search for news, standards, external context

For each user query:
1. Break it into independent sub-questions (1-4 sub-questions)
2. Assign each sub-question to the most appropriate retriever

Return ONLY valid JSON like:
{"plan": [{"retriever": "specs", "sub_question": "..."},
           {"retriever": "ops",   "sub_question": "..."}]}"""

def plan_query(question: str, verbose: bool = True) -> list[dict]:
    """Planner: decompose query into sub-questions with retriever assignments."""
    raw = generate(question, system=PLANNER_SYSTEM, max_tokens=1024)
    raw = raw.strip()
    if raw.startswith('```'):
        raw = raw.split('\n', 1)[1].rsplit('```', 1)[0]
    try:
        plan = json.loads(raw)["plan"]
    except (json.JSONDecodeError, KeyError):
        # Fallback: use all retrievers with the original question
        print("  Warning: planner output not valid JSON, falling back to all retrievers")
        plan = [{"retriever": r, "sub_question": question} for r in ALL_RETRIEVERS]
    
    if verbose:
        print("Planner decomposition:")
        for step in plan:
            print(f"  [{step['retriever']:8s}] {step['sub_question']}")
    
    return plan

# Test the planner
complex_q = """The Joint 4 problem is causing production delays at our customer. 
What is the current status of the fix, who owns it, what parts are involved, 
and are there any recent Slack updates I should know about?"""

plan = plan_query(complex_q)

## Step 5 — Parallel dispatch and aggregation

In [ ]:
def dispatch_parallel(plan: list[dict], verbose: bool = True) -> list[dict]:
    """Execute all retriever agents in parallel using ThreadPoolExecutor."""
    results = []
    
    def run_retriever(step):
        retriever_name = step["retriever"]
        sub_q = step["sub_question"]
        
        if retriever_name not in ALL_RETRIEVERS:
            return {"agent": retriever_name, "strategy": "error",
                    "context": f"Unknown retriever: {retriever_name}", "sub_question": sub_q}
        
        fn, _ = ALL_RETRIEVERS[retriever_name]
        t0 = time.time()
        result = fn(sub_q)
        result["sub_question"] = sub_q
        result["latency_ms"] = (time.time() - t0) * 1000
        return result
    
    t0 = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(run_retriever, step): step for step in plan}
        for future in concurrent.futures.as_completed(futures):
            results.append(future.result())
    total_ms = (time.time() - t0) * 1000
    
    if verbose:
        print(f"\nParallel dispatch completed in {total_ms:.0f} ms")
        for r in results:
            print(f"  [{r['agent']:20s}] {r['strategy']:20s} "
                  f"{r['latency_ms']:.0f}ms  {len(r['context'])} chars")
        sequential_ms = sum(r['latency_ms'] for r in results)
        print(f"  Sequential would take: {sequential_ms:.0f} ms  "
              f"(parallel is {sequential_ms/total_ms:.1f}× faster)")
    
    return results

retriever_results = dispatch_parallel(plan)

In [ ]:
# Synthesise all retriever outputs into a final answer
SYNTHESISER_SYSTEM = """You are the final synthesiser for Helios Robotics.
Multiple specialised agents have retrieved information relevant to a user's question.
Synthesise all their findings into a clear, complete answer.
- Lead with the most actionable information
- Reference specific part numbers, incident IDs, and contact names
- Note any conflicts or gaps between sources"""

def synthesise(question: str, retriever_results: list[dict]) -> str:
    """Final synthesis: combine all retriever outputs."""
    sections = []
    for r in retriever_results:
        sections.append(
            f"=== {r['agent']} (sub-question: {r['sub_question'][:60]}) ===\n"
            f"Strategy: {r['strategy']}\n"
            f"{r['context'][:800]}"
        )
    
    all_context = "\n\n".join(sections)
    prompt = f"Original question: {question}\n\nRetriever agent outputs:\n\n{all_context}\n\nSynthesised answer:"
    
    return generate(prompt, system=SYNTHESISER_SYSTEM, max_tokens=1500)

final_answer = synthesise(complex_q, retriever_results)
from ragkit.pretty import show_answer
show_answer(final_answer, title="Multi-Agent RAG — Final Answer")

## Step 6 — Full pipeline as one function

In [ ]:
def multi_agent_rag(question: str, verbose: bool = True) -> str:
    """Complete multi-agent RAG: plan → dispatch → synthesise."""
    if verbose:
        print(f"\n{'='*70}")
        print(f"Query: {question}")
        print(f"{'='*70}")
        print("\n[1] Planning...")
    
    plan = plan_query(question, verbose=verbose)
    
    if verbose:
        print("\n[2] Dispatching retrievers in parallel...")
    
    results = dispatch_parallel(plan, verbose=verbose)
    
    if verbose:
        print("\n[3] Synthesising...")
    
    answer = synthesise(question, results)
    
    if verbose:
        show_answer(answer)
    
    return answer

# Test with multiple complex queries
q1 = "I need to escalate the Joint 4 issue to the right person. Who owns the firmware fix, when will it be ready, and what part do I order for the interim hardware fix?"
_ = multi_agent_rag(q1)

In [ ]:
q2 = "What is Project Titan and how does it relate to the HeliosArm V3? Include who is leading it and what the current risks are."
_ = multi_agent_rag(q2)

## Step 7 — Compare vs single-agent router (NB06)

In [ ]:
# Rebuild the single-agent router from NB06 for comparison
from ragkit.data import build_chunked_corpus
from ragkit.vectorstore import build_collection, query_collection

full_texts, full_metas = build_chunked_corpus(chunk_size=200, overlap=40)
full_coll = build_collection("helios_compare", full_texts, full_metas, persist_dir="../.chroma")

SYSTEM_SIMPLE = "Answer precisely using the context. Include part numbers and names."

def simple_rag(question: str) -> str:
    hits = query_collection(full_coll, question, k=5)
    ctx = "\n\n".join(f"[{h.metadata['source']}]\n{h.text}" for h in hits)
    return generate(f"Context:\n{ctx}\n---\nQuestion: {question}", system=SYSTEM_SIMPLE)

compare_q = "Who should I contact about the Joint 4 firmware fix and what part number is the hardware interim fix?"

print("=" * 70)
print("NAIVE RAG:")
naive_ans = simple_rag(compare_q)
print(naive_ans)

print()
print("=" * 70)
print("MULTI-AGENT RAG:")
multi_ans = multi_agent_rag(compare_q, verbose=False)
print(multi_ans)

## Step 8 — Latency & cost analysis

In [ ]:
import matplotlib.pyplot as plt

q_bench = "What is the commissioning procedure for the HeliosArm V2?"

# Time naive RAG
t0 = time.time()
_ = simple_rag(q_bench)
naive_time = time.time() - t0

# Time multi-agent RAG components
t0 = time.time()
plan = plan_query(q_bench, verbose=False)
plan_time = time.time() - t0

t0 = time.time()
results = dispatch_parallel(plan, verbose=False)
dispatch_time = time.time() - t0

t0 = time.time()
_ = synthesise(q_bench, results)
synth_time = time.time() - t0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Timeline chart
ax = axes[0]
bars = ['Naive RAG', 'Multi-Agent: Plan', 'Multi-Agent: Dispatch', 'Multi-Agent: Synthesise']
times_ms = [naive_time*1000, plan_time*1000, dispatch_time*1000, synth_time*1000]
colors = ['#2b5797', '#e74c3c', '#f39c12', '#27ae60']
ax.barh(bars, times_ms, color=colors, alpha=0.85)
for i, v in enumerate(times_ms):
    ax.text(v + 20, i, f"{v:.0f}ms", va='center', fontsize=9)
ax.set_xlabel('Time (ms)')
ax.set_title('Latency breakdown')
ax.set_xlim(0, max(times_ms) * 1.3)

# LLM calls chart
ax2 = axes[1]
approaches = ['Naive RAG', 'Multi-Agent RAG']
llm_calls = [1, 2 + len(plan)]  # 1 for naive; planner + N retriever internal calls + synthesiser
ax2.bar(approaches, llm_calls, color=['#2b5797', '#e74c3c'], alpha=0.85, width=0.4)
for i, v in enumerate(llm_calls):
    ax2.text(i, v + 0.05, str(v), ha='center', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of LLM calls')
ax2.set_title('LLM call count (cost proxy)')
ax2.set_ylim(0, max(llm_calls) + 1)

plt.suptitle('Multi-Agent RAG vs Naive RAG', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

total_multi = (plan_time + dispatch_time + synth_time) * 1000
print(f"\nNaive RAG:       {naive_time*1000:.0f}ms, 1 LLM call")
print(f"Multi-Agent RAG: {total_multi:.0f}ms, ~{llm_calls[1]} LLM calls")
print(f"\nVerdict: Multi-Agent is {total_multi/naive_time/10:.1f}× slower and {llm_calls[1]}× more expensive")
print("         Worth it for complex, multi-domain questions; overkill for simple factual queries")

## Exercise — Decomposing a compound question for specialist agents

A multi-agent system starts with a **planner** that breaks a compound question into independent **sub-questions**, each assigned to the specialist agent that can answer it. A synthesiser then merges the results. This exercise checks the *plan*, not the prose.

The question mixes two domains:
> *"Compare the Spanish and French subjunctive, and name a Drag Race winner known for comedy."*

1. **Implement**: Complete `decompose` so it returns a list of `{"agent", "sub_question"}` items. The grammar comparison needs **two** language sub-questions (Spanish, French); the trivia part needs **one** drag sub-question.
2. **Check**: The plan should have 3 subtasks, use exactly the agents `{language_agent, dragrace_agent}`, and route 2 subtasks to the language agent.
3. **Reflect**: One sentence — which two subtasks here could run **in parallel**, and why does that matter for latency?

In [ ]:
def decompose(question: str) -> list[dict]:
    """Break a compound question into routed sub-questions."""
    return [
        {"agent": "language_agent", "sub_question": "How does the Spanish subjunctive work?"},
        {"agent": "language_agent", "sub_question": "How does the French subjunctive work?"},
        {"agent": "dragrace_agent", "sub_question": "Name a Drag Race winner known for comedy."},
    ]

question = "Compare the Spanish and French subjunctive, and name a Drag Race winner known for comedy."
plan = decompose(question)

for i, step in enumerate(plan, 1):
    print(f"  {i}. [{step['agent']:14s}] {step['sub_question']}")

agents      = {s["agent"] for s in plan}
lang_tasks  = sum(s["agent"] == "language_agent" for s in plan)

# ── Task 3: which subtasks run in parallel? (comment) ─────────────────────────
#   Your answer:

# ── Self-check ────────────────────────────────────────────────────────────────
assert len(plan) == 3, "the compound question splits into 3 sub-questions"
assert agents == {"language_agent", "dragrace_agent"}, "only language + dragrace agents are needed"
assert lang_tasks == 2, "the grammar comparison needs two language sub-questions"
print("\n✅ Exercise checks passed!")

## Summary of all 7 RAG patterns

| Pattern | Best for | Key ingredient | Complexity |
|---|---|---|---|
| 1. Naive RAG | Simple factual Q&A | Bi-encoder + vector DB | ★☆☆☆☆ |
| 2. Retrieve-and-Rerank | High accuracy requirements | Cross-encoder reranker | ★★☆☆☆ |
| 3. Multimodal RAG | Image + text corpora | CLIP, vision LLM | ★★★☆☆ |
| 4. Graph RAG | Multi-hop, connected domains | Neo4j + entity extraction | ★★★★☆ |
| 5. Hybrid RAG | Exact codes + semantic | BM25 + RRF | ★★☆☆☆ |
| 6. Agentic RAG (Router) | Mixed query types | LLM tool-calling | ★★★☆☆ |
| 7. Multi-Agent RAG | Complex, multi-domain | Planner + specialists + parallel dispatch | ★★★★★ |

**Production recommendation**: Start with Hybrid RAG (NB05) for most use cases. Add reranking (NB02) for quality-critical systems. Add the router (NB06) when query types are heterogeneous. Only reach for graph (NB04) or multi-agent (NB07) when your specific use case demands it.

## Exercises

1. **Add a real web search**: Replace `mock_web_search` with a real API call (Tavily, Brave, or SerpAPI). Does it improve answers?
2. **Error handling**: What happens if one retriever agent fails? Add a timeout and fallback to the dispatcher.
3. **Streaming**: Claude supports streaming output. Can you make the synthesiser stream its answer in real-time?
4. **Evaluate**: Define 10 benchmark questions and score all 7 patterns (correct/incorrect). Which pattern wins and on what question types?